# EdgeAttNet · Solar filament segmentation

Run all cells to load the selected model, inspect predictions, evaluate the fixed
validation split, and generate the competition CSV. Training is off by default.

**Setup:** install `requirements.txt`, restart the kernel, enable a CUDA GPU,
and attach the official competition data. Set the repository path below if
this notebook was uploaded into a separate Kaggle session. Split tables are
bundled; model artifacts download anonymously at a fixed revision.

The method uses grayscale input / 255, a DINOv3 ConvNeXt-Base encoder, an
edge-guided decoder, D4/TTA8 and a 31-feature CatBoost selector. The optional
55-epoch fine-tuning and OOF reconstruction branches are documented in
[the method guide](docs/METHOD.md) and [training guide](docs/TRAINING.md).


## 00 / Kaggle setup (automatic)

If only this notebook was uploaded to Kaggle, the next cell fetches the public repository into `/kaggle/working`. When the notebook is opened from a clone, or outside Kaggle, it does nothing.

In [ ]:
# 00 / Optional Kaggle setup: fetches the code only when this notebook was uploaded on its own.
# It does nothing when the notebook is opened from a clone of the repository, or outside Kaggle.
from pathlib import Path
import os, shutil, subprocess

REPO_URL = 'https://github.com/esraakh299/EdgeAttNet-DINOv3-Filament-Segmentation.git'
KAGGLE_WORKING = Path('/kaggle/working')
_has_code = lambda d: (d / 'src' / 'architecture.py').is_file()

_in_repo = any(_has_code(p) for p in (Path.cwd(), *Path.cwd().parents))
if not _in_repo and KAGGLE_WORKING.is_dir():
    _target = KAGGLE_WORKING / 'EdgeAttNet-DINOv3-Filament-Segmentation'
    if not _has_code(_target):
        if _target.exists():
            raise RuntimeError(f'{_target} exists but has no src/: delete it and re-run this cell.')
        if shutil.which('git') is None:
            raise RuntimeError('git is not available: upload the repository ZIP and set REPO_ROOT in the next cell.')
        _r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(_target)], capture_output=True, text=True)
        if _r.returncode != 0 or not _has_code(_target):
            raise RuntimeError('Could not fetch the code. Turn Internet on in the Kaggle session settings, '
                               'or upload the repository ZIP and set REPO_ROOT in the next cell.\n' + _r.stderr)
    os.chdir(_target)
    print('Code ready in', _target)
elif _in_repo:
    print('Repository code found: no setup needed.')
else:
    print('Not on Kaggle and not inside the repository: set REPO_ROOT in the next cell.')


## 01 / Reproducibility configuration

All run-dependent decisions are specified here. The inference path does not tune thresholds on the validation split.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform

# ── Execution mode ─────────────────────────────────────────────────────────
TRAIN_FROM_SCRATCH = False  # Default: strictly load the selected trained model
TRAIN_OOF_FROM_SCRATCH = False  # Rebuild the four held-out OOF networks (optional)
TRAIN_FILTER_FROM_SCRATCH = False  # Regenerate OOF maps and fit CatBoost (optional)
RUN_VALIDATION = True       # Computes multi-annotator pooled PQ over the 141-image split
SAVE_PREVIEW = True         # First sorted test image: input, probability and instance outlines
RUN_TEST = True             # Writes the Kaggle submission CSV for 180 test images

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p/'src'/'architecture.py').is_file()), Path.cwd())
# If needed: REPO_ROOT = Path('/kaggle/working/EdgeAttNet-DINOv3-Filament-Segmentation')
DATA_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else REPO_ROOT/'data'
SPLIT_ROOT = REPO_ROOT/'data'/'splits'
MODEL_DIR = Path(os.environ.get('EDGEATTNET_MODEL_DIR', str(REPO_ROOT/'models')))
OUT_DIR = Path('/kaggle/working/edgeattnet_results') if Path('/kaggle/working').exists() else REPO_ROOT/'outputs'

# Only required if TRAIN_FROM_SCRATCH=True. Training is DINOv3 fine-tuning;
# the encoder initialization itself comes from the official pretrained weights.
PRETRAINED_DINO_ENCODER = None  # e.g., Path('/kaggle/input/dinov3-encoder/model.pth')
OOF_PROBABILITIES_DIR = None    # Optional already-verified oof_index.json directory

# Public downloads can be disabled to require fully mounted/local model artifacts.
ALLOW_PUBLIC_DOWNLOAD = True
HF_REPO = 'youssefElghanam191/edgeattnet-dinov3-convnext-base'

assert (REPO_ROOT/'src'/'architecture.py').exists(), (
    'Run from the repository root: it must contain src/architecture.py. '
    'On Kaggle, clone/extract the public source first; no private credentials are used.')
OUT_DIR.mkdir(parents=True,exist_ok=True)
if str(REPO_ROOT) not in sys.path:sys.path.insert(0,str(REPO_ROOT))
print(f'Mode: {"TRAIN" if TRAIN_FROM_SCRATCH else "PRETRAINED"} | output: {OUT_DIR}')

## 02 / Data and environment

The curated split contains **566 training** and **141 validation** physical images. Validation uses the original COCO annotations, including all annotators for each physical image. The hidden test set contains **180 images**.

In [ ]:
import torch
from src.data import discover_dataset, image_folder, test_images, load_split, find_coco, AnnotationIndex
from src.artifacts import resolve_artifacts, load_model, load_filter, digest
from src.inference import run_validation,run_test

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required for 2048×2048 TTA8 inference.')
DEVICE=torch.device('cuda')
print('Python',platform.python_version(),'| PyTorch',torch.__version__,
      '| CUDA build',torch.version.cuda,'| GPU',torch.cuda.get_device_name(0))
DATASET=discover_dataset(DATA_ROOT)
print('Competition dataset:',DATASET)

## 03 / Model acquisition and optional training

**Default:** fetch the selected trained model and CatBoost filter and skip every training dependency. 

The optional fine-tuning branch uses the original **55-epoch DINOv3 ConvNeXt-Base EMA training recipe** and official pretrained encoder initialization. The OOF and filter switches are independent. If CatBoost is retrained, it uses only verified held-out maps from the four training-only fold models. No validation/test annotations are used to fit the selector.

In [ ]:
if TRAIN_FROM_SCRATCH or TRAIN_OOF_FROM_SCRATCH or TRAIN_FILTER_FROM_SCRATCH:
    if TRAIN_FROM_SCRATCH:
        if PRETRAINED_DINO_ENCODER is None:
            raise FileNotFoundError('Fine-tuning requires the licensed DINOv3 ConvNeXt-Base encoder state_dict (PRETRAINED_DINO_ENCODER).')
        subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'retrain_base.py'),
                        '--encoder',str(Path(PRETRAINED_DINO_ENCODER).resolve()),
                        '--output-root',str(OUT_DIR/'training'),'--data-root',str(DATASET)],
                       cwd=str(REPO_ROOT),check=True)
        snapshot=OUT_DIR/'training'/'dino_convnext_base'/'dice_best_ema.pth'
        if not snapshot.is_file():raise RuntimeError('Training has not produced a completed best EMA snapshot.')
        MODEL_DIR=OUT_DIR/'trained_artifacts';MODEL_DIR.mkdir(parents=True,exist_ok=True)
        subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'export_model.py'),
                        '--source',str(snapshot),'--output',str(MODEL_DIR/'edgeattnet_dinov3_base_best.pth')],check=True)

    if TRAIN_OOF_FROM_SCRATCH or TRAIN_FILTER_FROM_SCRATCH:
        # Existing verified index is valid only when the four OOF fold models are not being retrained.
        if TRAIN_OOF_FROM_SCRATCH or OOF_PROBABILITIES_DIR is None:
            pair_roots={}
            for pair in ('01','23'):
                out=OUT_DIR/f'DINO_OOF_Folds{pair}'
                cmd=[sys.executable,str(REPO_ROOT/'scripts'/'run_oof_pair.py'),
                     '--folds',pair,'--mode','train' if TRAIN_OOF_FROM_SCRATCH else 'pretrained',
                     '--data-root',str(DATASET),'--splits-root',str(SPLIT_ROOT),'--output',str(out),'--hf-repo',HF_REPO]
                if TRAIN_OOF_FROM_SCRATCH:
                    if PRETRAINED_DINO_ENCODER is None:
                        raise FileNotFoundError('Training OOF fold models requires PRETRAINED_DINO_ENCODER.')
                    cmd += ['--encoder',str(Path(PRETRAINED_DINO_ENCODER).resolve())]
                subprocess.run(cmd,cwd=str(REPO_ROOT),check=True)
                pair_roots[pair]=out
            verified_dir=OUT_DIR/'verified_oof'
            subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'audit_oof_coverage.py'),
                            '--pair-01',str(pair_roots['01']),'--pair-23',str(pair_roots['23']),
                            '--out',str(verified_dir)],cwd=str(REPO_ROOT),check=True)
            OOF_PROBABILITIES_DIR=verified_dir

    if TRAIN_FILTER_FROM_SCRATCH:
        # Train from leakage-safe maps. No in-sample final-model probabilities are allowed.
        assert OOF_PROBABILITIES_DIR is not None
        if not TRAIN_FROM_SCRATCH:
            MODEL_DIR=OUT_DIR/'refitted_filter_artifacts';MODEL_DIR.mkdir(parents=True,exist_ok=True)
            published=resolve_artifacts(None,hf_repo=HF_REPO,allow_download=ALLOW_PUBLIC_DOWNLOAD)
            import shutil
            shutil.copy2(published['weights'],MODEL_DIR/'edgeattnet_dinov3_base_best.pth')
        subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'fit_filter_oof.py'),
                        '--dataset',str(DATASET),'--splits',str(SPLIT_ROOT),
                        '--oof',str(OOF_PROBABILITIES_DIR),'--out',str(MODEL_DIR)],
                       cwd=str(REPO_ROOT),check=True)
    elif TRAIN_FROM_SCRATCH:
        # Fine-tuned Base model with the separately published, unchanged final filter.
        # This mode is not a claim that CatBoost was also retrained.
        import shutil
        published=resolve_artifacts(None,hf_repo=HF_REPO,allow_download=ALLOW_PUBLIC_DOWNLOAD)
        for artifact in ('regressor','config'):
            shutil.copy2(published[artifact],MODEL_DIR/published[artifact].name)


In [ ]:
paths=resolve_artifacts(MODEL_DIR,hf_repo=HF_REPO,
                        allow_download=ALLOW_PUBLIC_DOWNLOAD and not TRAIN_FROM_SCRATCH)
print('Weights:',paths['weights'].name,'SHA256:',digest(paths['weights']))
regressor,choice=load_filter(paths['regressor'],paths['config'])
net=load_model(paths['weights'],DEVICE)
print('Strict model state loaded | decoder floor/seed:',choice['low'],choice['seed'],
      '| min area:',choice['min_area'],'| filter cutoff:',choice['cutoff'])

## Visual review

The first sorted test image is chosen deterministically, without labels or quality-based selection. Orange outlines show accepted instances.

In [ ]:
if SAVE_PREVIEW:
    import matplotlib.pyplot as plt
    from src.inference import tta8_probability, select_instances
    preview_path = test_images(DATASET)[0]
    probability, raw = tta8_probability(net, preview_path, DEVICE)
    labels, kept, candidate_count = select_instances(probability, raw, regressor, choice)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    axes[0].imshow(raw, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('H-alpha input')
    axes[1].imshow(probability, cmap='magma', vmin=0, vmax=1)
    axes[1].set_title('Filament probability')
    axes[2].imshow(raw, cmap='gray', vmin=0, vmax=1)
    for label in kept:
        axes[2].contour(labels == label, levels=[0.5], colors=['#ff9b54'], linewidths=0.6)
    axes[2].set_title(f'{len(kept)} accepted / {candidate_count} candidates')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(preview_path.name)
    fig.savefig(OUT_DIR/'preview.png', dpi=160)
    plt.show()
    plt.close(fig)
    del probability, raw, labels


## 04 / Validation · original multi-annotator PQ

Every predicted instance is evaluated against each available annotator's instance labels. We sum IoU/TP/FP/FN across image–annotator comparisons, then calculate **PQ from pooled counts** (not an average of per-image PQ). A valid match requires **IoU > 0.5**. These are the original evaluator conventions.

In [ ]:
if RUN_VALIDATION:
    physical,annotation_sets=load_split(SPLIT_ROOT)
    idx=AnnotationIndex(find_coco(DATASET),annotation_sets)
    folder=image_folder(DATASET,'train')
    validation_files=[folder/name for name in physical.loc[physical.split=='val','file_name']]
    assert len(validation_files)==141 and all(p.is_file() for p in validation_files)
    metrics=run_validation(net,regressor,choice,validation_files,idx,DEVICE,OUT_DIR)
    print('Validation | PQ {pq:.6f} · SQ {sq:.6f} · RQ {rq:.6f} · TP {tp} · FP {fp} · FN {fn}'.format(**metrics))
else:
    print('Validation skipped by configuration (RUN_VALIDATION=False).')

## 05 / Test inference and Kaggle submission

The test path uses the *same* model, D4/TTA8 transforms, FP16 probability serialization, decoder settings and learned filter. No validation annotations or reference submission enter the prediction loop. Output: `submission.csv` containing `filament_id` and `segmentation_rle`.

In [ ]:
if RUN_TEST:
    submission_path,test_report=run_test(net,regressor,choice,test_images(DATASET),DEVICE,OUT_DIR)
    print('Submission:',submission_path)
    print('Test summary:',test_report)
else:
    print('Test export skipped by configuration (RUN_TEST=False).')

## 06 / Run provenance

The manifest captures the model and filter checksums, software environment and observed metrics for audit and later reruns.

In [ ]:
from importlib.metadata import distributions
from src.artifacts import DEFAULT_HF_REVISION

report={'mode':'train' if TRAIN_FROM_SCRATCH else 'pretrained',
        'architecture':'DINOv3 ConvNeXt-Base / EdgeAttNet',
        'torch':torch.__version__,'cuda':torch.version.cuda,
        'selected_decoder':choice,'model_sha256':digest(paths['weights']),
        'regressor_sha256':digest(paths['regressor']),
        'config_sha256':digest(paths['config'])}
if RUN_VALIDATION:report['validation']=metrics
if RUN_TEST:report['test']=test_report

report.update({
    'python': platform.python_version(),
    'platform': platform.platform(),
    'gpu': torch.cuda.get_device_name(0),
    'cudnn': torch.backends.cudnn.version(),
    'cudnn_benchmark': torch.backends.cudnn.benchmark,
    'cudnn_deterministic': torch.backends.cudnn.deterministic,
    'matmul_allow_tf32': torch.backends.cuda.matmul.allow_tf32,
    'hf_repo': HF_REPO,
    'default_hf_revision': DEFAULT_HF_REVISION if HF_REPO == 'youssefElghanam191/edgeattnet-dinov3-convnext-base' else None,
    'training_switches': {'base': TRAIN_FROM_SCRATCH, 'oof': TRAIN_OOF_FROM_SCRATCH, 'filter': TRAIN_FILTER_FROM_SCRATCH},
    'splits_sha256': {p.name: digest(p) for p in sorted(SPLIT_ROOT.glob('*.parquet'))},
    'packages': {d.metadata['Name']: d.version for d in distributions() if d.metadata['Name']},
})
(OUT_DIR/'run_manifest.json').write_text(json.dumps(report, indent=2, default=str))
(OUT_DIR/'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
print('Environment:', OUT_DIR/'environment.txt')


The supplied README selects **PyTorch 2.11.0 / CUDA 13.0**. Older CUDA 12.8
notes describe a historical single-image parity check. Full-dataset reproduction
must be confirmed against the selected submitted CSV. The checked-in package
pins are proposed; preserve this run's environment and manifest as evidence.
